# 02 - Assessing differential abundance using milestones

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import pertpy as pt
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

# Set Scanpy plotting parameters for consistent, high-quality figures
sc.settings.verbosity = 2  # Show progress
sc.settings.set_figure_params(
    dpi=150,  # High-res output
    dpi_save=300,  # High-res when saving
    format="pdf",  # or 'pdf', 'svg'
    facecolor="white",  # or 'none' for transparent
    frameon=False,  # No outer frame
    vector_friendly=True,  # No rasterization warnings
    fontsize=10,  # Base font size
    figsize=(4, 4),  # Default figure size
    transparent=True,  # Transparent background if saving
)

mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)


In [ ]:
settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

# Load and align all milestone measurements

In [ ]:
# Load abundance log fold changes for perturbed data
parse_norm_da_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/sccoda_tf_ko_significant_effects_eecs_milestones_pred_v3.csv", index_col=0)
parse_norm_da_df.columns = [col.upper() for col in parse_norm_da_df.columns]

# Load sccoda significant TFs
conditions = list(set(parse_norm_da_df.columns))
print(len(conditions))

In [ ]:
parse_norm_da_df

In [ ]:
# Check if PAX6 is in conditions
print("PAX6" in conditions)

In [ ]:
# Load normalized gene expression for VASA data
vasa_norm_exp_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.perturbed_tfs.expression.heatmap.csv", index_col=0)
vasa_norm_exp_df = vasa_norm_exp_df[[col for col in vasa_norm_exp_df.columns if col in conditions]]

In [ ]:
vasa_norm_exp_df

In [ ]:
# Check if PAX6 is in vasa_norm_exp_df columns
print("PAX6" in vasa_norm_exp_df.columns)

In [ ]:
# Load normalized imputed motif accessibility f
multiome_motif_acc_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/multiome.processed.eecs.perturbed_tfs.imputed_motif_acc.heatmap_v3.csv", index_col=0)
multiome_motif_acc_df = multiome_motif_acc_df[[col for col in multiome_motif_acc_df.columns if col in conditions]]

In [ ]:
# Check if PAX6 is in multiome_motif_acc_df columns
print("PAX6" in multiome_motif_acc_df.columns)

In [ ]:
# Load normalized imputed motif accessibility f
multiome_motif_acc_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/multiome.processed.eecs.perturbed_tfs.imputed_motif_acc.heatmap_v3.csv", index_col=0)
multiome_motif_acc_df = multiome_motif_acc_df[[col for col in multiome_motif_acc_df.columns if col in conditions]]

# Filter conditions to the ones for which 
# 1. we measured DA 
# 2. that are expressed in EECs
conditions = [c for c in conditions if c in vasa_norm_exp_df.columns.tolist()] 
print(len(conditions))

milestone_order = vasa_norm_exp_df.index.tolist()
parse_norm_da_df = parse_norm_da_df.reindex(milestone_order).copy()

# Subset to columns shared in both datasets (vasa_norm_exp_df, parse_norm_da_df, multiome_motif_acc_df)
shared_tfs = [tf for tf in parse_norm_da_df.columns if tf in vasa_norm_exp_df.columns]
shared_tfs = [tf for tf in shared_tfs if tf in multiome_motif_acc_df.columns]

vasa_norm_exp_df = vasa_norm_exp_df[shared_tfs]
parse_norm_da_df = parse_norm_da_df[shared_tfs]
multiome_motif_acc_df = multiome_motif_acc_df[shared_tfs]

## Assess consistency/directionality of effects among the three layers - DA, expression, motif acc

In [ ]:
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable

def hierarchy_pos(G, root=None, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
    if not nx.is_tree(G):
        raise TypeError("hierarchy_pos can only be used on a tree.")

    if root is None:
        root = next(iter(nx.topological_sort(G))) if isinstance(G, nx.DiGraph) else random.choice(list(G.nodes))

    def recurse(node, width, vert_loc, xcenter, pos, parent=None):
        pos[node] = (xcenter, vert_loc)
        children = list(G.neighbors(node))
        if parent is not None and not isinstance(G, nx.DiGraph):
            children = [c for c in children if c != parent]

        if children:
            dx = width / len(children)
            x = xcenter - width / 2 - dx / 2
            for child in children:
                x += dx
                recurse(child, width=dx, vert_loc=vert_loc - vert_gap, xcenter=x, pos=pos, parent=node)
        return pos

    return recurse(root, width, vert_loc, xcenter, pos={})

def get_norm(values, mode):
    v = np.array(values)
    
    if mode == "diverging":
        return TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2)
    else:
        return Normalize(vmin=v.min(), vmax=v.max())

def draw_modality_panel(G, pos, ax, tf, title, values_dict, cmap_name, mode):
    values = np.array(list(values_dict.values()))
    cmap = plt.cm.get_cmap(cmap_name)
    norm = get_norm(values, mode)

    # Prepare colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Draw edges
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        arrows=False,
        edge_color="gray",
        width=1.5,
        alpha=0.6
    )

    # Draw nodes
    node_colors = [values_dict[n] for n in G.nodes()]
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=900,
        node_color=node_colors,
        cmap=cmap,
        vmin=norm.vmin,
        vmax=norm.vmax,
        edgecolors="black",
        linewidths=0.7
    )

    # Labels
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)

    # Title
    ax.set_title(f"{tf} — {title}", fontsize=12, fontweight="bold")
    ax.axis("off")

    # Colorbar
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.03)
    cbar.ax.set_ylabel(title, rotation=90, fontsize=9)

def plot_tf_tree(G, pos, tf, modalities, figdir=None):
    fig, axes = plt.subplots(1, len(modalities), figsize=(6 * len(modalities), 6))

    for ax, (title, values_dict, cmap_name, mode) in zip(axes, modalities):
        draw_modality_panel(G, pos, ax, tf, title, values_dict, cmap_name, mode)

    plt.tight_layout()
    if figdir is not None:
        plt.savefig(figdir / f"{tf}_tree.pdf")
    plt.show()
    
    
edges = [
    ("NGN3+ cells", "EC/X/D/I/K cells"),
    ("EC/X/D/I/K cells", "EC cells"),
    ("EC/X/D/I/K cells", "X/D/I/K cells"),
    ("X/D/I/K cells", "X cells"),
    ("X/D/I/K cells", "D cells"),
    ("X/D/I/K cells", "I/K cells"),
]

G = nx.DiGraph()
G.add_edges_from(edges)
pos = hierarchy_pos(G, root="NGN3+ cells", width=2.0, vert_gap=0.5, vert_loc=0, xcenter=0.5)

In [ ]:
for tf in shared_tfs:

    da_s = dict(zip(parse_norm_da_df.index, parse_norm_da_df[tf].values))
    exp_s = dict(zip(parse_norm_da_df.index, vasa_norm_exp_df[tf].values))
    motif_s = dict(zip(parse_norm_da_df.index, multiome_motif_acc_df[tf].values))

    modalities = [
        ("Differential Abundance", da_s, "RdBu_r", "diverging"),
        ("Expression",              exp_s, "RdPu", "sequential"),
        ("Motif Accessibility",     motif_s, "BuGn", "sequential"),
    ]
    figdir = Path(sc.settings.figdir) / "tf_trees_v3"
    figdir.mkdir(exist_ok=True, parents=True)

    plot_tf_tree(G, pos, tf, modalities, figdir=figdir)

## Category per TF

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any
import networkx as nx


@dataclass
class TFModality:
    """A single modality layer for a TF."""
    title: str
    values: Dict[str, float]
    cmap: str = "viridis"
    mode: str = "sequential"   # or "diverging"


@dataclass
class TFNodeTree:
    """Container for a TF, its lineage tree, node positions, and modality layers."""
    tf: str
    graph: nx.DiGraph
    pos: Dict[str, Tuple[float, float]]
    modalities: List[TFModality] = field(default_factory=list)

    def add_modality(self, title: str, values: Dict[str, float],
                     cmap: str = "viridis", mode: str = "sequential"):
        """Add one modality panel."""
        self.modalities.append(TFModality(title, values, cmap, mode))

    def plot(self):
        """Plot the TF tree with all modalities using your existing helper funcs."""
        # we reuse your existing plot_tf_tree and draw_modality_panel
        modality_tuples = [
            (m.title, m.values, m.cmap, m.mode)
            for m in self.modalities
        ]
        plot_tf_tree(self.graph, self.pos, self.tf, modality_tuples)

In [ ]:
# Build tree
G = nx.DiGraph()
G.add_edges_from(edges)
pos = hierarchy_pos(G, root="NGN3+ cells", width=2.0, vert_gap=0.5)

# Create TF container
tf = "FOS"
tf_tree = TFNodeTree(tf=tf, graph=G, pos=pos)

# Build per-modality value dictionaries
modality_dicts = {
    "Perturbation Effect": dict(zip(parse_norm_da_df.index, parse_norm_da_df[tf])),
    "RNA": dict(zip(parse_norm_da_df.index, vasa_norm_exp_df[tf])),
    "ATAC": dict(zip(parse_norm_da_df.index, multiome_motif_acc_df[tf])),
}

# Add modalities
tf_tree.add_modality(
    title="RNA",
    values=modality_dicts["RNA"],
    cmap="RdPu",
    mode="sequential"
)

tf_tree.add_modality(
    title="ATAC",
    values=modality_dicts["ATAC"],
    cmap="BuGn",
    mode="sequential"
)

tf_tree.add_modality(
    title="Perturbation Effect",
    values=modality_dicts["Perturbation Effect"],
    cmap="RdBu",
    mode="diverging"
)

# Plot all three panels
tf_tree.plot()

In [ ]:
def modality_flow_signs(graph, values):
    """Return an array of sign(flow) for each directed edge in the graph."""
    signs = []
    for u, v in graph.edges:
        diff = values[v] - values[u]
        signs.append(np.sign(diff))
    return np.array(signs)


def alignment_score(signs_A, signs_B):
    """Compute average flow alignment between two modalities."""
    return np.mean(signs_A * signs_B)  # in [-1, +1]


def tf_flow_alignment(tf_tree, pert_name="Perturbation Effect", rna_name="RNA", atac_name="ATAC"):
    """Quantify flow alignment / inversion between modalities."""
    # extract modalities
    mod_rna  = next(m for m in tf_tree.modalities if m.title == rna_name)
    mod_atac = next(m for m in tf_tree.modalities if m.title == atac_name)
    mod_pert = next(m for m in tf_tree.modalities if m.title == pert_name)

    # compute signs for each modality
    s_rna  = modality_flow_signs(tf_tree.graph, mod_rna.values)
    s_atac = modality_flow_signs(tf_tree.graph, mod_atac.values)
    s_pert = modality_flow_signs(tf_tree.graph, mod_pert.values)

    # compute alignment
    align_rna  = alignment_score(s_rna, s_pert)
    align_atac = alignment_score(s_atac, s_pert)

    # combined score
    S = 0.5 * (align_rna + align_atac)

    if S < -0.33:
        label = "Activator"
    elif S > 0.33:
        label = "Repressor"
    else:
        label = "Mixed / Context-dependent"

    return dict(
        combined_score=S,
        align_rna=align_rna,
        align_atac=align_atac,
        label=label,
        signs=dict(
            rna=s_rna,
            atac=s_atac,
            pert=s_pert
        )
    )

In [ ]:
tf_flow_alignment(tf_tree)

In [ ]:
tf = "LMX1B"

# Build per-modality value dictionaries
modality_dicts = {
    "Differential Abundance": dict(zip(parse_norm_da_df.index, parse_norm_da_df[tf])),
    "Expression": dict(zip(parse_norm_da_df.index, vasa_norm_exp_df[tf])),
    "Motif Accessibility": dict(zip(parse_norm_da_df.index, multiome_motif_acc_df[tf])),
}

# Convert to your modality tuple structure
modalities = [
    ("Differential Abundance", modality_dicts["Differential Abundance"], "RdBu", "diverging"),
    ("Expression", modality_dicts["Expression"], "RdPu", "sequential"),
    ("Motif Accessibility", modality_dicts["Motif Accessibility"], "BuGn", "sequential"),
]

# Plot Gene Expression vs Differential Abundance
sns.scatterplot(
    x=[modality_dicts["Differential Abundance"][node] for node in G.nodes()],
    y=[modality_dicts["Expression"][node] for node in G.nodes()],
    s=100
)

# Label points
for node in G.nodes():
    plt.text(
        modality_dicts["Differential Abundance"][node],
        modality_dicts["Expression"][node],
        node,
        fontsize=9,
        ha='right'
    )

plt.grid(False)
plt.ylabel("Expression")
plt.xlabel("Differential Abundance (log2FC)")

In [ ]:
# Plot Gene Expression vs Differential Abundance
sns.scatterplot(
    x=[modality_dicts["Expression"][node] for node in G.nodes()],
    y=[modality_dicts["Differential Abundance"][node] for node in G.nodes()],
    s=100
)

# Label points
for node in G.nodes():
    plt.text(
        modality_dicts["Expression"][node],
        modality_dicts["Differential Abundance"][node],
        node,
        fontsize=9,
        ha='right'
    )

plt.grid(False)
plt.xlabel("Expression")
plt.ylabel("Differential Abundance (log2FC)")

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def get_value_vector(G, values_dict):
    """Return node values as a vector in consistent node order."""
    return np.array([values_dict[n] for n in G.nodes()])

def compute_modality_correlations(G, modalities):
    """
    modalities = [
        (name, values_dict, cmap, mode),
        ...
    ]

    Returns a dict mapping (mod1, mod2) → (pearson_r, pval)
    """
    results = {}
    names = [m[0] for m in modalities]

    # Precompute node-aligned vectors for each modality
    vectors = {
        name: get_value_vector(G, values_dict)
        for (name, values_dict, _, _) in modalities
    }

    # Compute pairwise correlations
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]

            # Skip modalities with NaNs
            if np.isnan(vectors[a]).any() or np.isnan(vectors[b]).any():
                results[(a, b)] = (np.nan, np.nan)
                continue

            corr, pval = stats.pearsonr(vectors[a], vectors[b])
            results[(a, b)] = (corr, pval)

    return results


tf_correlations = []

for tf in shared_tfs:

    # Build per-modality value dictionaries
    modality_dicts = {
        "Differential Abundance": dict(zip(parse_norm_da_df.index, parse_norm_da_df[tf])),
        "Expression": dict(zip(parse_norm_da_df.index, vasa_norm_exp_df[tf])),
        "Motif Accessibility": dict(zip(parse_norm_da_df.index, multiome_motif_acc_df[tf])),
    }

    # Convert to your modality tuple structure
    modalities = [
        ("Differential Abundance", modality_dicts["Differential Abundance"], "RdBu", "diverging"),
        ("Expression", modality_dicts["Expression"], "RdPu", "sequential"),
        ("Motif Accessibility", modality_dicts["Motif Accessibility"], "BuGn", "sequential"),
    ]

    # Compute correlations
    correlations = compute_modality_correlations(G, modalities)

    # Convert to tidy DataFrame
    df = (
        pd.DataFrame(correlations)
        .T.rename(columns={0: "pearson_r", 1: "p_value"})
        .reset_index()
        .rename(columns={"level_0": "modality_1", "level_1": "modality_2"})
    )
    df["TF"] = tf

    tf_correlations.append(df)

tf_correlations_df = pd.concat(tf_correlations, ignore_index=True).dropna()

In [ ]:
# Combine modality pairs into single column
tf_correlations_df["modality_pair"] = tf_correlations_df["modality_1"] + " vs. " + tf_correlations_df["modality_2"]
tf_correlations_df = tf_correlations_df.drop(columns=["modality_1", "modality_2"])

# Pivot wider for each modality pair
tf_correlations_pivot = tf_correlations_df.pivot_table(
    index="TF",
    columns="modality_pair",
    values="pearson_r"
).reset_index()

tf_correlations_pivot.rename(columns={
    "Differential Abundance vs. Expression": "corr_DA_EXP",
    "Differential Abundance vs. Motif Accessibility": "corr_DA_MOTIF",
    "Expression vs. Motif Accessibility": "corr_EXP_MOTIF"
}, inplace=True)

corr_df = tf_correlations_pivot.copy()
corr_df["abs_corr_DA_EXP"] = corr_df["corr_DA_EXP"].abs()
corr_df = corr_df.sort_values(by="abs_corr_DA_EXP", ascending=False)

# Group TFs based on in which "corner" they are (e.g. high DA-EXP, low EXP-MOTIF, etc)
def categorize_tf(row, threshold=0):
    da_exp = row["corr_DA_EXP"]
    exp_motif = row["corr_EXP_MOTIF"]
    
    if da_exp >= threshold and exp_motif >= threshold:
        return "Repressing - Open Chromatin"
    elif da_exp >= threshold and exp_motif <= -threshold:
        return "Repressing - Closed Chromatin"
    elif da_exp <= -threshold and exp_motif >= threshold:
        return "Activating - Open Chromatin"
    elif da_exp <= -threshold and exp_motif <= -threshold:
        return "Activating - Closed Chromatin"
    else:
        return "Unassigned"
    
corr_df["category"] = corr_df.apply(categorize_tf, axis=1, threshold=0)

# order tfs based on the group
category_order = [
    "Activating - Open Chromatin",
    "Repressing - Open Chromatin",
    "Activating - Closed Chromatin",
    "Repressing - Closed Chromatin",
    "Unassigned"
]
corr_df["category"] = pd.Categorical(corr_df["category"], categories=category_order, ordered=True)

In [ ]:
palette_category = {
    "Activating - Open Chromatin": "#2ca02c",   # green
    "Repressing - Open Chromatin": "#1f77b4",  # blue
    "Repressing - Closed Chromatin": "#ff7f0e",   # orange
    "Activating - Closed Chromatin": "#d62728",    # red
    "Unassigned": "#7f7f7f"               # gray
}

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=corr_df,
    x="corr_DA_EXP",
    y="corr_EXP_MOTIF",
    hue="category",
    palette=palette_category,
    s=60,
    edgecolor="black",
)

# Label points with TF names
for _, row in corr_df.iterrows():
    plt.text(
        row["corr_DA_EXP"],
        row["corr_EXP_MOTIF"],
        row["TF"],
        fontsize=8,
        ha="center",
        va="center"
    )

# Axes reference lines
plt.axvline(0, linestyle="--", color="black", linewidth=1)
plt.axhline(0, linestyle="--", color="black", linewidth=1)

plt.xlabel("Correlation: TF Differential Abundance  vs TF Expression")
plt.ylabel("Correlation: TF Expression vs TF Motif Acc.")

plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0)
plt.grid(False)
plt.tight_layout()
plt.show()